#### hugging face/timm

In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
import timm 
import torch  
import torch.nn as nn  
from safetensors.torch import load_file


In [7]:
inp = torch.randn(1, 2, 512, 512)
model = timm.create_model("efficientnet_b0", 
# model = timm.create_model("resnet34", 
# model = timm.create_model("mobilenetv3_large_100", 
                          features_only=True, 
                          in_chans=2, 
                          out_indices=(0, 1, 2, 3, 4))
features = model(inp)
for i, feat in enumerate(features):
    print(f"Stage {i}: shape={feat.shape}, channels={model.feature_info.channels()[i]}")   
model


Stage 0: shape=torch.Size([1, 16, 256, 256]), channels=16
Stage 1: shape=torch.Size([1, 24, 128, 128]), channels=24
Stage 2: shape=torch.Size([1, 40, 64, 64]), channels=40
Stage 3: shape=torch.Size([1, 112, 32, 32]), channels=112
Stage 4: shape=torch.Size([1, 320, 16, 16]), channels=320


EfficientNetFeatures(
  (conv_stem): Conv2d(2, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
    

### load pretrained weights to the model

In [3]:
encoder = timm.create_model('resnet18', 
                            features_only=True, 
                            in_chans=6, 
                            pretrained=True)


In [ ]:
model = timm.create_model(
    model_name='efficientnet_b0',
    in_chans=6, 
    features_only=True,
    pretrained=True,
    )


Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.


In [ ]:
inp = torch.randn(1, 6, 512, 512)
features = model(inp)
for i, feat in enumerate(features):
    print(f"Stage {i}: shape={feat.shape}, channels={model.feature_info.channels()[i]}")   


Stage 0: shape=torch.Size([1, 16, 256, 256]), channels=16
Stage 1: shape=torch.Size([1, 24, 128, 128]), channels=24
Stage 2: shape=torch.Size([1, 40, 64, 64]), channels=40
Stage 3: shape=torch.Size([1, 112, 32, 32]), channels=112
Stage 4: shape=torch.Size([1, 320, 16, 16]), channels=320


#### hugging face/transformers

In [ ]:
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
from PIL import Image
import requests

processor = SegformerImageProcessor.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")
model = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b0-finetuned-ade-512-512")

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

inputs = processor(images=image, return_tensors="pt")
outputs = model(**inputs)
logits = outputs.logits  # shape (batch_size, num_labels, height/4, width/4)


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

In [ ]:
inputs['pixel_values'].shape 


torch.Size([1, 3, 512, 512])

In [9]:
import torch
import torch.nn as nn
from transformers import SegformerModel

def create_mit_encoder(model_name="nvidia/mit-b0", in_chans=3):
    """
    加载 SegFormer 编码器，并自适应修改输入通道数 (模拟 timm 的 in_chans 行为)
    """
    # 1. 加载预训练模型 (仅包含 encoder 部分)
    encoder = SegformerModel.from_pretrained(model_name)
    
    # 2. 如果输入通道不是 3 (例如 6通道的光学，或 1通道的DEM)，修改第一层卷积
    if in_chans != 3:
        old_conv = encoder.encoder.patch_embeddings[0].proj
        new_conv = nn.Conv2d(in_chans, old_conv.out_channels, 
                             kernel_size=old_conv.kernel_size, 
                             stride=old_conv.stride, 
                             padding=old_conv.padding, 
                             bias=(old_conv.bias is not None))
        
        # 权重初始化策略：复用预训练的 RGB 权重，多出的通道随机初始化或复制
        with torch.no_grad():
            if in_chans > 3: # 例如 6 通道
                new_conv.weight[:, :3, :, :] = old_conv.weight
                new_conv.weight[:, 3:, :, :] = old_conv.weight[:, :in_chans-3, :, :]
            elif in_chans == 1: # 例如 DEM 1 通道
                new_conv.weight[:, 0:1, :, :] = old_conv.weight.sum(dim=1, keepdim=True) # RGB取均值
                
            if old_conv.bias is not None:
                new_conv.bias = old_conv.bias
                
        # 替换原有的第一层
        encoder.encoder.patch_embeddings[0].proj = new_conv
        
    return encoder

# ----------------- 测试代码 -----------------
if __name__ == '__main__':
    # 1. 初始化你的两个分支的 MiT 编码器
    encoder_opt = create_mit_encoder("nvidia/mit-b0", in_chans=6)
    encoder_dem = create_mit_encoder("nvidia/mit-b0", in_chans=1)
    
    # 2. 模拟输入数据 (Batch=2, H=W=256)
    x_opt = torch.randn(2, 6, 256, 256)
    x_dem = torch.randn(2, 1, 256, 256)
    
    # 3. 提取特征 (设置 output_hidden_states=True 相当于 timm 的 features_only)
    outputs_opt = encoder_opt(pixel_values=x_opt, output_hidden_states=True)
    
    # hidden_states 是一个 tuple，包含了 4 个尺度的特征图 (1/4, 1/8, 1/16, 1/32)
    feas_opt = outputs_opt.hidden_states 
    
    print("MiT 提取的光学特征图尺度：")
    for i, feat in enumerate(feas_opt):
        print(f"Stage {i+1}: {feat.shape}") 
  
  

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerModel LOAD REPORT from: nvidia/mit-b0
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiT 提取的光学特征图尺度：
Stage 1: torch.Size([2, 32, 64, 64])
Stage 2: torch.Size([2, 64, 32, 32])
Stage 3: torch.Size([2, 160, 16, 16])
Stage 4: torch.Size([2, 256, 8, 8])
